# Lab 6: Multi-Agent Coding Team for ERP Development
## CPE494: ERP Systems with AI Integration

This notebook is the local VS Code orchestrator for the **CPE494-agent-coding-team** repository.

It coordinates four Gemini-powered agents:

1. **Architect**: creates the sprint blueprint and task list.
2. **Coder**: generates source code and project artifacts.
3. **Logic Tester**: audits code for data, transaction, security, and compilation risks.
4. **UI Auditor**: audits UI/UX against the Zen Green reference rules.

The generated application lives in the companion repository:

```text
../CPE494-erp-invoice-app-by-ai
```

This notebook intentionally uses plain Python first. Later, students can convert the workflow to use ADK and MCP.

This version also creates a **separate run folder for every sprint attempt**. Each run folder stores the input prompt/spec snapshot, Gemini calls, generated files, audit logs, applied files, and command logs. This lets us know exactly which specs and prompts produced the final code, instead of relying on human memory, which is adorable but not version control.


## Step 0: Expected folder structure

The two repositories should be cloned side-by-side inside the parent folder `lab-coding-agent`:

```text
lab-coding-agent/
  CPE494-agent-coding-team/
    notebooks/
      agent_coding_workflow.ipynb
    prompts/
      role_architect.txt
      role_coder.txt
      role_logic_tester.txt
      role_ui_auditor.txt
    logic_specs/
      erp_invoice_logic.md
    ui_specs/
      erp_invoice_ui.md
      ui_reference.pdf
    outputs/
      runs/
        <run_id>/
          run_manifest.json
          input_snapshot/
          gemini_calls.jsonl
          architect_manifest.md
          tasks.json
          generated_files/
          audit_logs.jsonl
          applied_files.json
          command_logs.jsonl
    .env
    .gitignore

  CPE494-erp-invoice-app-by-ai/
    Pages/
    Models/
    Data/
    wwwroot/
```

The `.env` file belongs in the root of `CPE494-agent-coding-team` and should contain:

```text
GOOGLE_API_KEY=your_api_key_here
```

Recommended `.gitignore` entries for the agent repo:

```text
.env
.venv/
__pycache__/
.ipynb_checkpoints/
outputs/runs/
```

The `outputs/runs/` folder is generated runtime evidence, not source code. You may keep selected run folders for teaching or debugging, but by default they should not be committed unless your assignment requires students to submit their run logs.


In [1]:
# Step 1: Imports and path setup
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

from IPython.display import clear_output

# The notebook is expected to run from the CPE494-agent-coding-team repo root.
# In VS Code, set the notebook working directory to the repository root if needed.
REPO_ROOT = Path.cwd().resolve()

# If the notebook is opened from notebooks/, move one level up automatically.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.resolve()

TARGET_APP_PATH = (REPO_ROOT.parent / "CPE494-erp-invoice-app-by-ai").resolve()

PROMPTS_DIR = REPO_ROOT / "prompts"
LOGIC_SPECS_DIR = REPO_ROOT / "logic_specs"
UI_SPECS_DIR = REPO_ROOT / "ui_specs"
OUTPUTS_DIR = REPO_ROOT / "outputs"
RUNS_DIR = OUTPUTS_DIR / "runs"

OUTPUTS_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# These are assigned when start_agentic_workflow() creates a new run.
RUN_ID = None
RUN_DIR = None
INPUT_SNAPSHOT_DIR = None
GENERATED_DIR = None

print("Agent repo root:", REPO_ROOT)
print("Target app path:", TARGET_APP_PATH)
print("Runs folder:", RUNS_DIR)


Agent repo root: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team
Target app path: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-erp-invoice-app-by-ai
Runs folder: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\runs


In [2]:
# Step 2: Validate required folders and files
required_paths = [
    PROMPTS_DIR / "role_architect.txt",
    PROMPTS_DIR / "role_coder.txt",
    PROMPTS_DIR / "role_logic_tester.txt",
    PROMPTS_DIR / "role_ui_auditor.txt",
    PROMPTS_DIR / "ui_checklist_for_coder.txt",
    LOGIC_SPECS_DIR / "erp_invoice_logic.md",
    UI_SPECS_DIR / "erp_invoice_ui.md",
]

missing = [p for p in required_paths if not p.exists()]

if not TARGET_APP_PATH.exists():
    missing.append(TARGET_APP_PATH)

if missing:
    print("Missing required paths:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Fix the missing files/folders before continuing.")

print("All required folders and files were found.")


All required folders and files were found.


In [3]:
'''
# Step 3: Load environment variables and initialize Gemini client
load_dotenv(REPO_ROOT / ".env")

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY was not found. Create a .env file in the agent repo root.")

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
client = genai.Client(api_key=api_key)

print("Gemini client initialized.")
print("Model:", MODEL_NAME)
'''
print("We use the above if we plan to use API keys, but we decided to use gcloud authentication for better security and ease of use in GCP environments. So we can skip this step for now.")


We use the above if we plan to use API keys, but we decided to use gcloud authentication for better security and ease of use in GCP environments. So we can skip this step for now.


In [4]:
#New Version 

import vertexai
from vertexai.generative_models import GenerativeModel
from google.oauth2 import service_account

# Clears the cell's output area immediately
clear_output(wait=True)

# 1. Path to the key file you downloaded from Cloud Shell
KEY_FILE = "../dev-key.json"  
PROJECT_ID = "glassy-grin-494714-f5" 
LOCATION = "us-central1" # us-central1 has most models. Other choices: asia-southeast1 (2, 3)
MODEL_NAME = "gemini-2.5-pro" #Also try "gemini-2.5-flash", gemini-2.0-flash, gemini-2.0-flash-live

# 2. Load credentials from the JSON file
credentials = service_account.Credentials.from_service_account_file(KEY_FILE)

# 3. Initialize Vertex AI with your project and credentials
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials
)

print("Vertex AI initialized successfully!")

# Initialize the model (Gemini 1.5 Flash is fast and cost-effective)
model = GenerativeModel(MODEL_NAME)


# Send a prompt
response = model.generate_content("If today were Tuesday what would tomorrow be?")

# Print the result
print("Model: ", MODEL_NAME, ". Response: ", response.text)

Vertex AI initialized successfully!


c:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Model:  gemini-2.5-pro . Response:  Tomorrow would be Wednesday.


In [5]:
# Step 4: Load agent prompts and specification documents

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

ROLE_FILES = {
    "architect": PROMPTS_DIR / "role_architect.txt",
    "coder": PROMPTS_DIR / "role_coder.txt",
    "logic_tester": PROMPTS_DIR / "role_logic_tester.txt",
    "ui_auditor": PROMPTS_DIR / "role_ui_auditor.txt",
    "ui_checklist_for_coder": PROMPTS_DIR / "ui_checklist_for_coder.txt",
}

SPEC_FILES = {
    "logic_specs": LOGIC_SPECS_DIR / "erp_invoice_logic.md",
    "ui_specs": UI_SPECS_DIR / "erp_invoice_ui.md",
}

ARCHITECT_PROMPT = read_text(ROLE_FILES["architect"])
CODER_PROMPT = read_text(ROLE_FILES["coder"])
LOGIC_TESTER_PROMPT = read_text(ROLE_FILES["logic_tester"])
UI_AUDITOR_PROMPT = read_text(ROLE_FILES["ui_auditor"])
UI_CHECKLIST_FOR_CODER = read_text(ROLE_FILES["ui_checklist_for_coder"])

LOGIC_SPECS = read_text(SPEC_FILES["logic_specs"])
UI_SPECS = read_text(SPEC_FILES["ui_specs"])

print("Loaded prompts and specs.")
print("Logic spec length:", len(LOGIC_SPECS))
print("UI spec length:", len(UI_SPECS))
print("UI checklist length:", len(UI_CHECKLIST_FOR_CODER))


Loaded prompts and specs.
Logic spec length: 7010
UI spec length: 7661
UI checklist length: 755


In [6]:
# Step 5: Runtime logging and utility functions

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def make_run_id(sprint_id: str) -> str:
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    clean_sprint_id = re.sub(r"[^A-Za-z0-9_\-]+", "_", sprint_id.strip())
    return f"{timestamp}_{clean_sprint_id}"


def save_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def get_git_commit_hash(repo_path: Path) -> str:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=str(repo_path),
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return "unknown"


def get_git_status_short(repo_path: Path) -> str:
    try:
        result = subprocess.run(
            ["git", "status", "--short"],
            cwd=str(repo_path),
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return "unknown"


def require_run_dir() -> Path:
    if RUN_DIR is None:
        raise RuntimeError("No run has been initialized. Call start_agentic_workflow() first.")
    return RUN_DIR


def snapshot_input_files(run_dir: Path):
    """Copy the exact prompt/spec files used for this run into input_snapshot/."""
    snapshot_dir = run_dir / "input_snapshot"
    snapshot_dir.mkdir(parents=True, exist_ok=True)

    input_files = {**ROLE_FILES, **SPEC_FILES}
    snapshot_records = []

    for label, src in input_files.items():
        dst = snapshot_dir / src.name
        shutil.copy2(src, dst)
        snapshot_records.append({
            "label": label,
            "source_path": str(src.relative_to(REPO_ROOT)),
            "snapshot_path": str(dst.relative_to(run_dir)),
            "sha256": file_sha256(src),
        })

    save_json(run_dir / "input_snapshot_manifest.json", {"files": snapshot_records})
    return snapshot_records


def initialize_run(sprint_id: str, sprint_goal: str) -> Path:
    """Create a new run folder and record the repo state and input snapshots."""
    global RUN_ID, RUN_DIR, INPUT_SNAPSHOT_DIR, GENERATED_DIR

    RUN_ID = make_run_id(sprint_id)
    RUN_DIR = RUNS_DIR / RUN_ID
    INPUT_SNAPSHOT_DIR = RUN_DIR / "input_snapshot"
    GENERATED_DIR = RUN_DIR / "generated_files"

    RUN_DIR.mkdir(parents=True, exist_ok=False)
    INPUT_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
    GENERATED_DIR.mkdir(parents=True, exist_ok=True)

    input_records = snapshot_input_files(RUN_DIR)

    manifest = {
        "run_id": RUN_ID,
        "sprint_id": sprint_id,
        "sprint_goal": sprint_goal.strip(),
        "created_at": now_iso(),
        "model": MODEL_NAME,
        "agent_repo_root": str(REPO_ROOT),
        "target_app_path": str(TARGET_APP_PATH),
        "agent_repo_commit": get_git_commit_hash(REPO_ROOT),
        "target_app_repo_commit": get_git_commit_hash(TARGET_APP_PATH),
        "agent_repo_status_short": get_git_status_short(REPO_ROOT),
        "target_app_repo_status_short": get_git_status_short(TARGET_APP_PATH),
        "input_files": input_records,
    }
    save_json(RUN_DIR / "run_manifest.json", manifest)
    (OUTPUTS_DIR / "latest_run.txt").write_text(RUN_ID, encoding="utf-8")

    print("Created run folder:", RUN_DIR)
    print("Run ID:", RUN_ID)
    return RUN_DIR

def call_agent(agent_name: str, system_prompt: str, user_prompt: str, temperature: float = 0.2, metadata: dict | None = None) -> str:
    """Call Gemini and log the full prompt and response into the current run folder."""
    run_dir = require_run_dir()
    metadata = metadata or {}
    started_at = now_iso()

    response_text = ""
    status = "success"
    error_message = None

    try:
        from vertexai.generative_models import GenerativeModel
        
        # 1. Initialize the model specifically for this agent with their system prompt
        agent_model = GenerativeModel(
            model_name=MODEL_NAME,
            system_instruction=system_prompt
        )

        # 2. Call generate_content with the user prompt and temperature
        response = agent_model.generate_content(
            contents=user_prompt,
            generation_config={
                "temperature": temperature,
            }
        )
        response_text = response.text or ""
        return response_text
        
    except Exception as e:
        status = "error"
        error_message = str(e)
        raise
    finally:
        append_jsonl(run_dir / "gemini_calls.jsonl", {
            "run_id": RUN_ID,
            "agent": agent_name,
            "model": MODEL_NAME,
            "temperature": temperature,
            "started_at": started_at,
            "finished_at": now_iso(),
            "status": status,
            "error_message": error_message,
            "metadata": metadata,
            "system_prompt": system_prompt,
            "user_prompt": user_prompt,
            "response_text": response_text,
        })

def strip_markdown_code_fence(text: str) -> str:
    """Remove accidental markdown fences from coder output."""
    cleaned = text.strip()
    fence = re.match(r"^```[a-zA-Z0-9_+-]*\s*(.*?)\s*```$", cleaned, re.DOTALL)
    if fence:
        return fence.group(1).strip()
    return cleaned


def starts_with_pass(text: str) -> bool:
    """Auditors must start with PASS to approve."""
    return text.strip().upper().startswith("PASS")


def safe_relative_path(file_name: str) -> Path:
    """Return a safe relative path and prevent path traversal."""
    raw = file_name.replace("\\", "/").strip().lstrip("/")
    rel = Path(raw)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe file path from task: {file_name}")
    return rel


def get_project_tree(root: Path, max_files: int = 120) -> str:
    """Return a compact project tree for agent context."""
    ignore_dirs = {".git", "bin", "obj", ".vs", ".vscode", "node_modules"}
    files = []
    for path in root.rglob("*"):
        if any(part in ignore_dirs for part in path.parts):
            continue
        if path.is_file():
            try:
                rel = path.relative_to(root)
                files.append(str(rel).replace("\\", "/"))
            except ValueError:
                pass
        if len(files) >= max_files:
            break
    return "\n".join(sorted(files))


def read_existing_target_file(relative_path: Path) -> str:
    """Read existing target file if it exists."""
    target_file = TARGET_APP_PATH / relative_path
    if target_file.exists() and target_file.is_file():
        return target_file.read_text(encoding="utf-8", errors="replace")
    return ""


def save_generated_file(relative_path: Path, content: str) -> Path:
    """Save generated file under the current run's generated_files folder."""
    run_dir = require_run_dir()
    destination = run_dir / "generated_files" / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
    return destination


def append_audit_log(record: dict):
    run_dir = require_run_dir()
    record = {"timestamp": now_iso(), "run_id": RUN_ID, **record}
    append_jsonl(run_dir / "audit_logs.jsonl", record)


In [7]:
# Step 6: Architect agent

def extract_tasks_from_manifest(manifest_text: str) -> list[dict]:
    """Extract JSON task list from the Architect response."""
    json_match = re.search(r"```json\s*(.*?)\s*```", manifest_text, re.DOTALL | re.IGNORECASE)
    if json_match:
        raw_json = json_match.group(1)
    else:
        # Fallback: find the first JSON object in the response.
        obj_match = re.search(r"\{.*\}", manifest_text, re.DOTALL)
        raw_json = obj_match.group(0) if obj_match else ""

    if not raw_json:
        return []

    try:
        parsed = json.loads(raw_json)
    except json.JSONDecodeError as e:
        print("Could not parse Architect JSON:", e)
        return []

    def build_task_description(task: dict) -> str:
        """Support both legacy task_description schema and new engineering-contract schema."""
        direct = (task.get("task_description") or "").strip()
        if direct:
            return direct

        purpose = (task.get("purpose") or "").strip()
        acceptance = task.get("acceptance_criteria") or []
        dependencies = task.get("dependencies") or []
        verification = task.get("verification_methods") or []
        test_expectations = task.get("test_expectations") or []
        risk_notes = task.get("risk_notes") or []
        done_definition = (task.get("done_definition") or "").strip()

        lines = []
        if purpose:
            lines.append(f"Purpose: {purpose}")
        if acceptance:
            lines.append("Acceptance Criteria:")
            lines.extend([f"- {item}" for item in acceptance])
        if dependencies:
            lines.append("Dependencies:")
            lines.extend([f"- {item}" for item in dependencies])
        if verification:
            lines.append("Verification Methods:")
            for method_item in verification:
                method = method_item.get("method", "") if isinstance(method_item, dict) else str(method_item)
                rule = method_item.get("rule", "") if isinstance(method_item, dict) else ""
                cmd = method_item.get("command", "") if isinstance(method_item, dict) else ""
                summary = f"- {method}" if method else "- verification"
                if rule:
                    summary += f": {rule}"
                if cmd:
                    summary += f" (command: {cmd})"
                lines.append(summary)
        if test_expectations:
            lines.append("Test Expectations:")
            lines.extend([f"- {item}" for item in test_expectations])
        if risk_notes:
            lines.append("Risk Notes:")
            lines.extend([f"- {item}" for item in risk_notes])
        if done_definition:
            lines.append(f"Done Definition: {done_definition}")

        return "\n".join(lines).strip()

    tasks = parsed.get("tasks", [])
    valid_tasks = []
    for i, task in enumerate(tasks, start=1):
        file_name = task.get("file_name")
        task_description = build_task_description(task)
        task_id = task.get("task_id") or i
        feature_group = task.get("feature_group")
        if file_name and task_description:
            valid_tasks.append({
                "task_id": task_id,
                "feature_group": feature_group,
                "file_name": file_name,
                "task_description": task_description,
                "task_contract": task,
            })
    return valid_tasks


def call_architect(sprint_goal: str) -> tuple[str, list[dict]]:
    """Ask the Architect to create a sprint plan and task list."""
    run_dir = require_run_dir()
    project_tree = get_project_tree(TARGET_APP_PATH)

    user_prompt = f"""
You are planning work for the target ASP.NET Core project below.

TARGET PROJECT PATH:
{TARGET_APP_PATH}

CURRENT TARGET PROJECT TREE:
{project_tree}

LOGIC SPECIFICATIONS:
{LOGIC_SPECS}

UI SPECIFICATIONS:
{UI_SPECS}

UI CHECKLIST (MUST SATISFY):
{UI_CHECKLIST_FOR_CODER}

SPRINT GOAL:
{sprint_goal}

Important orchestration requirement:
- The JSON task list must use file paths relative to the target app root.
- Example file_name values: Pages/Index.cshtml, Pages/Index.cshtml.cs, wwwroot/css/zen-green.css, Models/product.cs.
- Do not write code. Produce a human-readable manifest and a JSON task list only.
"""
    manifest = call_agent(
        agent_name="architect",
        system_prompt=ARCHITECT_PROMPT,
        user_prompt=user_prompt,
        temperature=0.15,
        metadata={"step": "create_manifest_and_tasks"},
    )
    tasks = extract_tasks_from_manifest(manifest)

    (run_dir / "architect_manifest.md").write_text(manifest, encoding="utf-8")
    save_json(run_dir / "tasks.json", {"tasks": tasks})

    return manifest, tasks


In [8]:
# Step 7: Coder and auditor loop

def build_coder_prompt(relative_path: Path, task_description: str, previous_feedback: str = "") -> str:
    existing_content = read_existing_target_file(relative_path)
    project_tree = get_project_tree(TARGET_APP_PATH)

    return f"""
Generate the complete content for this target file:

FILE PATH:
{relative_path.as_posix()}

TASK DESCRIPTION:
{task_description}

CURRENT TARGET PROJECT TREE:
{project_tree}

EXISTING FILE CONTENT, IF ANY:
{existing_content if existing_content else "[File does not currently exist.]"}

LOGIC SPECIFICATIONS:
{LOGIC_SPECS}

UI SPECIFICATIONS:
{UI_SPECS}

PREVIOUS AUDIT FEEDBACK TO FIX:
{previous_feedback if previous_feedback else "[None]"}

Return only the raw file content. Do not include markdown fences or explanations.
"""


def audit_generated_file(relative_path: Path, code_text: str, task_description: str) -> tuple[str, str]:
    logic_prompt = f"""
Review this generated file for logic, data, security, compilation, and architecture risks.

FILE PATH:
{relative_path.as_posix()}

TASK DESCRIPTION:
{task_description}

LOGIC SPECIFICATIONS:
{LOGIC_SPECS}

CODE TO REVIEW:
{code_text}
"""

    ui_prompt = f"""
Review this generated file for UI/UX, Zen Green theme, English-only UI, responsive behavior, tooltips, alignment, and validation display rules.

FILE PATH:
{relative_path.as_posix()}

TASK DESCRIPTION:
{task_description}

UI SPECIFICATIONS:
{UI_SPECS}

CODE TO REVIEW:
{code_text}
"""

    logic_feedback = call_agent(
        agent_name="logic_tester",
        system_prompt=LOGIC_TESTER_PROMPT,
        user_prompt=logic_prompt,
        temperature=0.0,
        metadata={"step": "logic_audit", "file_name": relative_path.as_posix()},
    )
    ui_feedback = call_agent(
        agent_name="ui_auditor",
        system_prompt=UI_AUDITOR_PROMPT,
        user_prompt=ui_prompt,
        temperature=0.0,
        metadata={"step": "ui_audit", "file_name": relative_path.as_posix()},
    )
    return logic_feedback, ui_feedback


def orchestrate_file_build(task: dict, max_attempts: int = 2) -> dict:
    """Generate and audit one file. Save to this run's generated_files only if both audits pass."""
    relative_path = safe_relative_path(task["file_name"])
    task_description = task["task_description"]

    print(f"\nBuilding: {relative_path.as_posix()}")
    previous_feedback = ""

    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}")

        coder_prompt = build_coder_prompt(relative_path, task_description, previous_feedback)
        generated_code = call_agent(
            agent_name="coder",
            system_prompt=CODER_PROMPT,
            user_prompt=coder_prompt,
            temperature=0.2,
            metadata={
                "step": "generate_file",
                "file_name": relative_path.as_posix(),
                "attempt": attempt,
            },
        )
        generated_code = strip_markdown_code_fence(generated_code)

        logic_feedback, ui_feedback = audit_generated_file(relative_path, generated_code, task_description)
        logic_passed = starts_with_pass(logic_feedback)
        ui_passed = starts_with_pass(ui_feedback)
        print(f"Logic Tester: {'PASS' if logic_passed else 'FAIL'}")
        print(f"UI Auditor: {'PASS' if ui_passed else 'FAIL'}")

        record = {
            "task_id": task.get("task_id"),
            "file_name": relative_path.as_posix(),
            "attempt": attempt,
            "logic_passed": logic_passed,
            "ui_passed": ui_passed,
            "logic_feedback": logic_feedback,
            "ui_feedback": ui_feedback,
        }
        append_audit_log(record)

        # Save every attempt for traceability. Only PASS output is copied to generated_files/.
        attempt_dir = require_run_dir() / "attempts" / relative_path.as_posix().replace("/", "__")
        attempt_dir.mkdir(parents=True, exist_ok=True)
        (attempt_dir / f"attempt_{attempt:02d}_code.txt").write_text(generated_code, encoding="utf-8")
        save_json(attempt_dir / f"attempt_{attempt:02d}_audit.json", record)

        if logic_passed and ui_passed:
            saved_path = save_generated_file(relative_path, generated_code)
            print(f"PASS. Saved to {saved_path}")
            return {
                "task_id": task.get("task_id"),
                "file_name": relative_path.as_posix(),
                "status": "PASS",
                "attempts": attempt,
                "saved_path": str(saved_path),
            }

        previous_feedback = f"""
LOGIC TESTER FEEDBACK:
{logic_feedback}

UI AUDITOR FEEDBACK:
{ui_feedback}
"""
        print("Audit failed. Feedback will be sent to the Coder for the next attempt.")
        if not logic_passed:
            print("Logic Tester feedback:")
            print(logic_feedback)
        if not ui_passed:
            print("UI Auditor feedback:")
            print(ui_feedback)

    print("FAILED after maximum attempts:", relative_path.as_posix())
    return {
        "task_id": task.get("task_id"),
        "file_name": relative_path.as_posix(),
        "status": "FAIL",
        "attempts": max_attempts,
    }


In [9]:
# Step 8: Full sprint workflow

# Demo safety toggle:
# - False: ask for human confirmation before generation
# - True: continue automatically (useful for non-interactive demo runs)
AUTO_APPROVE = False

def start_agentic_workflow(sprint_id: str, sprint_goal: str, max_attempts_per_file: int = 2):
    """Start a complete sprint run with traceable runtime artifacts."""
    run_dir = initialize_run(sprint_id=sprint_id, sprint_goal=sprint_goal)

    print("Architect is preparing the sprint plan...")
    manifest, tasks = call_architect(sprint_goal)

    print("" + "=" * 80)
    print("ARCHITECT MANIFEST")
    print("=" * 80)
    print(manifest)
    print("=" * 80)

    if not tasks:
        print("No valid tasks were found. Check the Architect prompt or JSON output.")
        save_json(run_dir / "workflow_result.json", {
            "run_id": RUN_ID,
            "status": "NO_TASKS",
            "finished_at": now_iso(),
        })
        return []

    print(f"Architect proposed {len(tasks)} tasks:")
    for task in tasks:
        print(f"{task['task_id']}. {task['file_name']}")

    print("\n" + "=" * 80)
    print("HUMAN CONFIRMATION REQUIRED")
    print("- Enter 'y' to continue code generation and audits")
    print("- Enter 'n' to cancel this run before code generation")
    print(f"- AUTO_APPROVE is currently: {AUTO_APPROVE}")
    print("=" * 80)

    if AUTO_APPROVE:
        approval = "y"
        print("AUTO_APPROVE=True -> proceeding without interactive prompt.")
    else:
        try:
            approval = input("Proceed with code generation? (y/n): ").strip().lower()
        except EOFError:
            approval = "n"
            print("No interactive input available. Defaulting to 'n' and cancelling safely.")

    if approval != "y":
        print("Sprint cancelled before code generation.")
        save_json(run_dir / "workflow_result.json", {
            "run_id": RUN_ID,
            "status": "CANCELLED_BEFORE_CODE_GENERATION",
            "finished_at": now_iso(),
        })
        return []

    results = []
    for task in tasks:
        result = orchestrate_file_build(task, max_attempts=max_attempts_per_file)
        results.append(result)

    passed = sum(1 for r in results if r.get("status") == "PASS")
    failed = sum(1 for r in results if r.get("status") != "PASS")

    save_json(run_dir / "workflow_result.json", {
        "run_id": RUN_ID,
        "status": "COMPLETE",
        "finished_at": now_iso(),
        "passed_files": passed,
        "failed_files": failed,
        "results": results,
    })

    print(f"Sprint generation complete. PASS: {passed}, FAIL: {failed}")
    print(f"Run folder: {run_dir}")
    print(f"Generated files are in: {run_dir / 'generated_files'}")
    return results


## Step 9: Sprint 1 goal

Run the next cell to start Sprint 1. The workflow will:

1. Create a new run folder under `outputs/runs/<run_id>/`.
2. Snapshot all role prompts and spec files used in the run.
3. Record the current Git commit hash and Git status for both repos.
4. Ask the Architect for a manifest and JSON task list.
5. Ask for your approval.
6. Generate each file.
7. Audit each file with the Logic Tester and UI Auditor.
8. Save every Gemini call to `gemini_calls.jsonl`.
9. Save every generated attempt and audit result.
10. Save passing files to `generated_files/`.

After generation finishes, open `notebooks/transfer_generated_files.ipynb` to review the latest run and choose which files to apply to the ASP.NET Core app repo. That transfer notebook reads `outputs/latest_run.txt`, so it can be run separately after restarting VS Code or the kernel.

The target app repo does not need a special logging folder. It should remain the actual application codebase. Git history in that repo records what was applied and committed.


In [10]:
SPRINT_1_ID = "sprint_01"
SPRINT_1_GOAL = """
Sprint 1: Infrastructure and app shell.

Build the English-only ASP.NET Core Razor Pages app shell for the ERP Invoice application.
Create or update only the files needed for:

- global Zen Green CSS tokens and common UI classes
- main layout with English-only menu placeholders
- landing page with welcome message
- login placeholder with database selector, username, password, remember-me, and sign-in button
- menu placeholders for Master Data, Transactions, Reports, and Control

Do not implement full authentication yet. Do not implement product, customer, or invoice CRUD yet.
All UI text must be English only.
"""



In [11]:
SPRINT_2_ID = "sprint_02_customer_form_foundation"
SPRINT_2_GOAL = """
Sprint 2: Customer form from logic spec (no authentication).

Deliver a working Customer form flow quickly, without authentication, and verify responsive attribute-value UI rules.
From the Welcome page, users must be able to open a Customer form and save customer records to LocalDB.

Create or update only the files needed for:

- add a visible hyperlink/button on the Welcome page to open the Customer form
- configure EF Core with SQL Server LocalDB connection
- implement Customer entity and table fields exactly from logic_specs/erp_invoice_logic.md (do not invent or rename fields)
- include id primary key and rowversion concurrency behavior exactly per logic spec
- enforce server-side validation:
  - required fields
  - code uniqueness
  - credit_limit >= 0
- build one working Customer form page (English-only UI) with Save action
- UI layout test requirements for Customer form:
  - place Is Active at the top
  - show the first 9 customer fields as regular attribute-value fields in a responsive 4-columns-per-row layout
  - show phone_number, email, and credit_limit as tabular attribute-value fields with widths 12, 20, and 14 characters respectively
  - show delivery address fields as regular attribute-value fields in a responsive 3-columns-per-row layout
- include a Cancel/Back action to return to the Welcome page
- show English success/validation feedback
- keep Zen Green styling consistent; right-align numeric/currency fields; mark required fields

Out of scope for this sprint:
- authentication/authorization
- role management
- product/invoice modules
- advanced search/filter/reporting

All UI text must be English only.
"""


In [14]:

results = start_agentic_workflow(SPRINT_2_ID, SPRINT_2_GOAL, max_attempts_per_file=3)


Created run folder: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\runs\2026-05-04_212642_sprint_02_customer_form_foundation
Run ID: 2026-05-04_212642_sprint_02_customer_form_foundation
Architect is preparing the sprint plan...
ARCHITECT MANIFEST
Here is the technical execution plan for Sprint 2.

### Technical Blueprint: Sprint 2 - Customer Module

This sprint focuses on establishing the data foundation for the `Customer` entity and building the complete UI flow for creating, listing, and editing customer records. The work is divided into four main phases: Data Foundation, UI Foundation & Navigation, Customer List View, and Customer Form (Upsert) View. All work must adhere to the project's architectural and language standards.

---

#### **Phase 1: Data Foundation**

This phase establishes the database connection, the Entity Framework Core context, and the core `Customer` entity model.

*   **Task 1: Configure Database Connectio

In [13]:
# Stop here during normal execution.
# Open notebooks/transfer_generated_files.ipynb when you are ready to review and apply files.
raise SystemExit("Normal stop point: review results, then use transfer_generated_files.ipynb to apply files.")


SystemExit: Normal stop point: review results, then use transfer_generated_files.ipynb to apply files.

c:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
